# Week 10 Practice: Calling a Real API

Run each cell in order. Everything here uses APIs that need **no key and no signup**, so it works straight out of the box.

If `requests` isn't installed yet, run the next cell once.

In [ ]:
# Run once, then comment it out
# !pip install requests

In [ ]:
import requests
import json
import time

## 1. Your first request

`JSONPlaceholder` is a practice API that returns fake blog posts and users.

In [ ]:
response = requests.get("https://jsonplaceholder.typicode.com/posts/1", timeout=5)

print("Status code:", response.status_code)
print("OK?", response.ok)

In [ ]:
# .json() turns the response body into Python objects
post = response.json()

print(type(post))
print(post)

In [ ]:
# It's just a dictionary - use it like any other
print("Title:", post["title"])
print("Body:", post["body"][:60], "...")

## 2. Look before you leap

When you don't know the shape of a response, print it with indentation first.

In [ ]:
response = requests.get("https://jsonplaceholder.typicode.com/users/1", timeout=5)
user = response.json()

print(json.dumps(user, indent=2)[:600])

In [ ]:
# Nested values need a chain of keys
print(user["name"])
print(user["address"]["city"])
print(user["company"]["name"])

## 3. Query parameters

Let `requests` build the URL for you with `params=`. Never glue strings together by hand.

In [ ]:
params = {"userId": 1}

response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params=params,
    timeout=5
)

print("Requested:", response.url)

posts = response.json()
print("Got", len(posts), "posts")

In [ ]:
for post in posts[:5]:
    print(f"#{post['id']}: {post['title']}")

## 4. Real, live data

Open-Meteo returns actual current weather and needs no key. Change the coordinates and re-run.

In [ ]:
params = {
    "latitude": 41.90,          # River Forest, IL
    "longitude": -87.81,
    "current": "temperature_2m,wind_speed_10m",
    "temperature_unit": "fahrenheit"
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=5)
response.raise_for_status()

current = response.json()["current"]
print(f"{current['temperature_2m']}F, wind {current['wind_speed_10m']} mph")

## 5. When things go wrong

Networks fail. Wrap requests in `try` / `except` and always pass a `timeout`.

In [ ]:
def fetch(url, **kwargs):
    """Fetch a URL and return parsed JSON, or None on any failure."""
    try:
        response = requests.get(url, timeout=5, **kwargs)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print("Timed out")
    except requests.exceptions.ConnectionError:
        print("Could not reach the server")
    except requests.exceptions.HTTPError as e:
        print("Server returned an error:", e)
    except ValueError:
        print("Response was not valid JSON")
    return None


# A URL that does not exist - should print an error, not crash
print(fetch("https://jsonplaceholder.typicode.com/posts/999999"))

## 6. Looping politely

Pause between calls so you don't hammer the server.

In [ ]:
users = []

for user_id in range(1, 6):
    data = fetch(f"https://jsonplaceholder.typicode.com/users/{user_id}")
    if data:
        users.append(data)
    time.sleep(0.5)          # be a good citizen

for u in users:
    print(u["name"], "-", u["address"]["city"])

## 7. Save what you fetched

Writing results to a file means you can analyze them later without calling the API again.

In [ ]:
import csv

with open("users.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["id", "name", "email", "city"])
    writer.writeheader()
    for u in users:
        writer.writerow({
            "id": u["id"],
            "name": u["name"],
            "email": u["email"],
            "city": u["address"]["city"]
        })

print("Saved", len(users), "users to users.csv")

---

## Your Turn

Work through these in the empty cells below. Add cells as you need them.

**Exercise 1.** Fetch all comments on post 1 (`https://jsonplaceholder.typicode.com/comments` with `params={"postId": 1}`) and print each commenter's name and email.

**Exercise 2.** Fetch all 100 posts. Build a dictionary counting how many posts each `userId` wrote, then print the results sorted from most to fewest.

**Exercise 3.** Write a function `get_todo_summary(user_id)` that fetches that user's todos from `https://jsonplaceholder.typicode.com/todos` and returns a string like `"User 3: 8 of 20 completed"`.

**Exercise 4.** Pick two cities, fetch the current temperature for each from Open-Meteo, and print which one is warmer. Handle the case where either request fails.

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4